# 研究与工程思维 1/6：从现象到可证伪假设

这不是一节“记术语”的课，而是一节**改变判断过程**的实验课。

| 项目 | 内容 |
|---|---|
| 核心问题 | 识别率下降时，怎样避免凭第一印象修错地方？ |
| 迁移价值 | 适用于线上故障、性能回退、学习困难和任何‘为什么’问题。 |
| 建议投入 | 90～150 分钟；先预测，再运行，再保留被推翻的判断 |
| 通关证据 | 能把结论写成“主张—证据—反证—边界—下一步” |

固定闭环：

```text
观察（发生了什么） → 假设（可能为什么） → 区分性预测 → 最小实验
        ↑                                      ↓
        └──── 更新置信度、记录反例、决定下一步 ────┘
```

**观察不是原因，总分不是解释，相关不是干预效果，运行成功不是结论成立。**


## 课前预测：先暴露自己的判断规则

1. 用一句话回答：识别率下降时，怎样避免凭第一印象修错地方？
2. 写出你最可能犯的判断错误，例如“只看平均值”或“看到相关就认定因果”。
3. 为本课写一个可被数据推翻的预测；不要写“应该会更好”这种没有阈值的话。
4. 写出什么结果会让你改变主意。

完成实验后回来修正。保留原答案，因为“怎样改主意”本身就是思维能力证据。


## 一手资料与课程取舍

- [NIST：实验设计的定义与目标](https://itl.nist.gov/div898/handbook/pri/section1/pri11.htm)
- [NIST：随机化、重复与设计原则](https://www.itl.nist.gov/div898/handbook/pmd/section3/pmd33.htm)

课程把这些资料转成小型、确定性、可运行的 ASR 实验。示例数据用于理解方法，不代表真实产品结论。


## 1. 把五种句子分开

| 类型 | 例子 | 能否直接推出原因 |
|---|---|---|
| 观察 | 噪声切片 WER 从 12% 变成 25% | 不能 |
| 假设 | 新降噪器损坏了辅音 | 不能，等待检验 |
| 预测 | 若假设成立，关闭降噪器后噪声切片至少改善 5 个百分点 | 可以检验 |
| 结果 | 配对样本改善 1.1 个百分点，区间跨 0 | 证据不足 |
| 决策 | 暂不发布，先检查重采样与 VAD | 还包含成本和风险 |

“系统变差了，因为最近换了前端”把时间先后误当成原因。先列竞争假设，再找能让它们给出不同预测的测试。


In [1]:
from dataclasses import dataclass

@dataclass(frozen=True)
class Hypothesis:
    name: str
    prediction: str
    falsifier: str
    test_cost: float
    discrimination: float
    coverage: float

hypotheses = [
    Hypothesis("降噪损伤", "关闭降噪后 noisy WER 改善 >= 5pp", "改善 < 1pp", 2, 0.90, 0.65),
    Hypothesis("采样率误配", "8 kHz 输入的频带/时长合同异常", "全部输入合同一致", 1, 0.95, 0.40),
    Hypothesis("VAD 截断", "删除错误集中在句首/句尾", "边界与中间删除率相同", 1, 0.80, 0.55),
    Hypothesis("说话人偏移", "新说话人切片独立变差", "同说话人配对也退化", 3, 0.70, 0.80),
]

def test_value(h: Hypothesis) -> float:
    # 区分力和覆盖越高越好，成本越低越好；只是排序启发式，不是概率。
    if h.test_cost <= 0:
        raise ValueError("test_cost must be positive")
    return h.discrimination * h.coverage / h.test_cost

for h in sorted(hypotheses, key=test_value, reverse=True):
    print(f"{h.name:8s} value={test_value(h):.3f} | {h.prediction}")


VAD 截断   value=0.440 | 删除错误集中在句首/句尾
采样率误配    value=0.380 | 8 kHz 输入的频带/时长合同异常
降噪损伤     value=0.293 | 关闭降噪后 noisy WER 改善 >= 5pp
说话人偏移    value=0.187 | 新说话人切片独立变差


## 2. 好假设必须冒险

“可能是数据问题”几乎不会失败，所以信息量很低。把它改写成：

> 若主要原因是 VAD 截断，那么新版相对旧版新增的删除错误中，至少 60% 位于首尾 300 ms；在关闭 VAD、其余条件固定的配对实验中差异应显著缩小。

它包含对象、方向、阈值、对照和反证。阈值应在看结果前写；看完数据再移动门槛叫事后合理化。


In [2]:
records = [
    {"slice": "clean", "position": "middle", "old": 1, "new": 1},
    {"slice": "clean", "position": "edge",   "old": 0, "new": 1},
    {"slice": "noisy", "position": "middle", "old": 2, "new": 3},
    {"slice": "noisy", "position": "edge",   "old": 1, "new": 5},
    {"slice": "noisy", "position": "edge",   "old": 2, "new": 6},
]

extra_by_position = {}
for row in records:
    extra = max(0, row["new"] - row["old"])
    extra_by_position[row["position"]] = extra_by_position.get(row["position"], 0) + extra

edge_fraction = extra_by_position["edge"] / sum(extra_by_position.values())
print("新增错误按位置:", extra_by_position)
print(f"首尾占比={edge_fraction:.1%}")
print("预测通过?", edge_fraction >= 0.60)
assert edge_fraction >= 0.60


新增错误按位置: {'middle': 1, 'edge': 9}
首尾占比=90.0%
预测通过? True


## 3. 证据强度阶梯

```text
传闻 < 单个例子 < 同分布统计 < 固定样本配对对照
     < 随机化/重复实验 < 独立复现 < 目标场景持续监控
```

强证据也只支持特定范围。一次 8 kHz 中文短句实验不能推出所有语言、设备和长音频都成立。

### 练习：先不要运行答案

把“幅值低于 1000，所以声音小”改写为至少三个竞争假设，并为每个假设写一个区分性测试。必须考虑位深/dtype、RMS 与 peak、麦克风增益或标定基准。


In [3]:
def validate_hypothesis_card(card: dict) -> list[str]:
    required = ["observation", "hypothesis", "prediction", "threshold", "falsifier", "scope"]
    missing = [key for key in required if not str(card.get(key, "")).strip()]
    return missing

example = {
    "observation": "int16 文件的 peak=850",
    "hypothesis": "录音链路增益过低",
    "prediction": "同设备同距离的校准音 RMS 比基准低至少 12 dB",
    "threshold": "delta_rms_db <= -12",
    "falsifier": "换算到 dBFS 并校准后与基准差小于 3 dB",
    "scope": "当前设备、距离和 int16 PCM；不外推到 float 音频",
}
print("缺失字段:", validate_hypothesis_card(example))
assert validate_hypothesis_card(example) == []


缺失字段: []


## 闭卷挑战

选一个你最近遇到的技术问题，写 3 个竞争假设。设计一个成本不超过 10 分钟、却能最大程度区分它们的实验；解释为什么它比直接改代码更有信息量。

回答时强制使用下面的证据卡：

```text
主张：
证据：
最强替代解释：
什么结果会推翻主张：
适用边界：
下一步最小实验：
```


## 最小掌握门禁

- [ ] 我在运行前写了方向和数量级预测。
- [ ] 我能指出示例结论中至少一个替代解释。
- [ ] 我能从空白重写本课核心函数，并用边界输入测试。
- [ ] 我能说明“没有发现差异”和“证明没有差异”的区别。
- [ ] 我把一次被数据推翻的判断写入 `LEARNING_LOG.md`。
- [ ] 我能把本课方法迁移到一个非 ASR 问题。

下一步：第 2 课把‘平均 WER’拆成错误类型与数据切片。
